In [1]:
import pandas as pd

ranked = pd.read_csv(r"C:\tri_disease_project\data\processed\shared_genes_ranked.csv")
gene_list = ranked['hgnc_symbol'].tolist()

print("Total genes for network analysis:", len(gene_list))

# save as plain text, one gene per line — this is what STRING/Cytoscape stringApp wants
with open(r"C:\tri_disease_project\data\processed\shared_genes_for_string.txt", "w") as f:
    f.write("\n".join(gene_list))

print("Saved gene list for STRING import.")

Total genes for network analysis: 690
Saved gene list for STRING import.


In [2]:
import pandas as pd

node_table = pd.read_csv(r"C:\tri_disease_project\results\tables\node_table_with_clusters.csv")

print("Columns:", node_table.columns.tolist())
print("\nCluster size distribution:")
cluster_sizes = node_table['__ccCluster'].value_counts()
print(cluster_sizes.head(10))

largest_cluster_id = cluster_sizes.idxmax()
largest_cluster_size = cluster_sizes.max()
print(f"\nLargest cluster ID: {largest_cluster_id}, size: {largest_cluster_size} nodes")

lcc_genes = node_table.loc[node_table['__ccCluster'] == largest_cluster_id]
print("\nColumns available to identify gene names:")
print(lcc_genes.columns.tolist()[:20])

Columns: ['@id', '__ccCluster', 'compartment::cytoskeleton', 'compartment::cytosol', 'compartment::endoplasmic reticulum', 'compartment::endosome', 'compartment::extracellular', 'compartment::golgi apparatus', 'compartment::lysosome', 'compartment::mitochondrion', 'compartment::nucleus', 'compartment::peroxisome', 'compartment::plasma membrane', 'display name', 'name', 'query term', 'selected', 'shared name', 'stringdb::canonical name', 'stringdb::database identifier', 'stringdb::description', 'stringdb::enhancedLabel Passthrough', 'stringdb::full name', 'stringdb::imageurl', 'stringdb::namespace', 'stringdb::node color', 'stringdb::node type', 'stringdb::sequence', 'stringdb::species', 'stringdb::STRING style', 'stringdb::structures', 'target::development level', 'target::family', 'tissue::adrenal gland', 'tissue::blood', 'tissue::bone', 'tissue::bone marrow', 'tissue::eye', 'tissue::gall bladder', 'tissue::heart', 'tissue::intestine', 'tissue::kidney', 'tissue::liver', 'tissue::lung'

In [3]:
# 'display name' or 'query term' should hold the actual gene symbols — let's check both
print(lcc_genes[['name', 'display name', 'query term']].head(10))

lcc_gene_symbols = lcc_genes['display name'].dropna().unique().tolist()
print(f"\nTotal unique gene symbols in LCC: {len(lcc_gene_symbols)}")
print(sorted(lcc_gene_symbols)[:30])

# save for later phases (Phase 5 enrichment, Phase 8 features) and for re-selecting in Cytoscape
with open(r"C:\tri_disease_project\data\processed\lcc_gene_list.txt", "w") as f:
    f.write("\n".join(sorted(lcc_gene_symbols)))

import pandas as pd
pd.Series(sorted(lcc_gene_symbols), name='hgnc_symbol').to_csv(
    r"C:\tri_disease_project\data\processed\lcc_gene_list.csv", index=False)

print("\nSaved LCC gene list.")

                     name display name   query term
269  9606.ENSP00000482245        DACH1        DACH1
272  9606.ENSP00000497406       SETBP1       SETBP1
273  9606.ENSP00000501146        HNF4G        HNF4G
274  9606.ENSP00000468785         MBD1         MBD1
275  9606.ENSP00000491657          LPP          LPP
276  9606.ENSP00000460885  CORO7-PAM16  CORO7-PAM16
277  9606.ENSP00000469696        PDE4C        PDE4C
278  9606.ENSP00000411397        MARK3        MARK3
281  9606.ENSP00000391269       RBFOX1       RBFOX1
282  9606.ENSP00000480090       PKD1L3       PKD1L3

Total unique gene symbols in LCC: 374
['ABCC8', 'ABHD5', 'ACAN', 'ACE', 'ACE2', 'ACHE', 'ACVR1C', 'ACVR2A', 'ADAMTS9', 'ADAMTSL3', 'ADARB2', 'ADCY5', 'ADCY9', 'ADH1B', 'ADH4', 'ADIPOQ', 'ADRA1D', 'ADRA2A', 'ADRA2B', 'ADRA2C', 'ADRB1', 'ADRB2', 'AFF1', 'AGER', 'AGMO', 'AGTR1', 'AK1', 'AKR1B1', 'ALDH1A2', 'ALDH2']

Saved LCC gene list.


In [4]:
import pandas as pd
import networkx as nx

edges = pd.read_csv(r"C:\tri_disease_project\results\tables\lcc_edges.csv")
print("Edge table columns:", edges.columns.tolist())
print(edges.head())

Edge table columns: ['EdgeBetweenness', 'interaction', 'name', 'selected', 'shared interaction', 'shared name', 'stringdb::coexpression', 'stringdb::cooccurrence', 'stringdb::databases', 'stringdb::experiments', 'stringdb::fusion', 'stringdb::interspecies', 'stringdb::neighborhood', 'stringdb::score', 'stringdb::textmining']
   EdgeBetweenness interaction  \
0      2692.062954          pp   
1       128.689160          pp   
2       683.522838          pp   
3       323.801148          pp   
4      3002.397774          pp   

                                             name  selected  \
0  9606.ENSP00000269305 (pp) 9606.ENSP00000287820     False   
1  9606.ENSP00000377372 (pp) 9606.ENSP00000480132     False   
2  9606.ENSP00000269305 (pp) 9606.ENSP00000384371     False   
3  9606.ENSP00000378323 (pp) 9606.ENSP00000379330     False   
4  9606.ENSP00000269305 (pp) 9606.ENSP00000384863     False   

  shared interaction                                     shared name  \
0                

In [5]:
import pandas as pd
import networkx as nx
import re

edges = pd.read_csv(r"C:\tri_disease_project\results\tables\lcc_edges.csv")

# parse "9606.ENSP00000269305 (pp) 9606.ENSP00000287820" into source/target
def parse_edge(name):
    match = re.match(r"(\S+)\s+\(pp\)\s+(\S+)", name)
    if match:
        return match.group(1), match.group(2)
    return None, None

edges[['source', 'target']] = edges['name'].apply(lambda x: pd.Series(parse_edge(x)))
edges = edges.dropna(subset=['source', 'target'])

print("Parsed edges:", len(edges))
print(edges[['source', 'target']].head())

# build the graph
G = nx.from_pandas_edgelist(edges, source='source', target='target')

print("\n--- Independent networkx verification ---")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Average degree:", sum(dict(G.degree()).values()) / G.number_of_nodes())
print("Number of connected components:", nx.number_connected_components(G))
print("Average clustering coefficient:", nx.average_clustering(G))

if nx.is_connected(G):
    print("Diameter:", nx.diameter(G))
    print("Radius:", nx.radius(G))
    print("Characteristic path length (avg shortest path):", nx.average_shortest_path_length(G))
else:
    print("Graph is NOT fully connected — diameter/radius/path length undefined for disconnected graphs")
    largest_cc = max(nx.connected_components(G), key=len)
    print("Largest component size:", len(largest_cc))

print("Network density:", nx.density(G))

Parsed edges: 983
                 source                target
0  9606.ENSP00000269305  9606.ENSP00000287820
1  9606.ENSP00000377372  9606.ENSP00000480132
2  9606.ENSP00000269305  9606.ENSP00000384371
3  9606.ENSP00000378323  9606.ENSP00000379330
4  9606.ENSP00000269305  9606.ENSP00000384863

--- Independent networkx verification ---
Number of nodes: 374
Number of edges: 983
Average degree: 5.25668449197861
Number of connected components: 1
Average clustering coefficient: 0.28636467505714447
Diameter: 12
Radius: 6
Characteristic path length (avg shortest path): 4.241745638055368
Network density: 0.014092987914151768


In [7]:
import pandas as pd

hubba = pd.read_csv(r"C:\tri_disease_project\results\tables\cytohubba_all_scores.csv")
print("Shape:", hubba.shape)
print(hubba.columns.tolist())
print(hubba.head())

Shape: (374, 13)
['node_name', 'MCC', 'DMNC', 'MNC', 'Degree', 'EPC', 'BottleNeck', 'EcCentricity', 'Closeness', 'Radiality', 'Betweenness', 'Stress', 'ClusteringCoefficient']
                      node_name      MCC  DMNC   MNC  Degree   EPC  \
9606.ENSP00000383996        2.0  0.00000   1.0   2.0   4.666   2.0   
9606.ENSP00000233242   363517.0  0.60089  15.0  16.0  85.134  11.0   
9606.ENSP00000354947        2.0  0.00000   1.0   2.0   2.294   2.0   
9606.ENSP00000376902        1.0  0.00000   1.0   1.0  18.021   1.0   
9606.ENSP00000352798        1.0  0.00000   1.0   1.0   1.857   1.0   

                      BottleNeck  EcCentricity  Closeness   Radiality  \
9606.ENSP00000383996     0.11111      77.25595    7.82842   744.00000   
9606.ENSP00000233242     0.11111     128.69802    9.64879  1904.09198   
9606.ENSP00000354947     0.10000      67.58611    7.11260   145.86009   
9606.ENSP00000376902     0.12500     115.86786    9.51743     0.00000   
9606.ENSP00000352798     0.10000      

In [8]:
node_table = pd.read_csv(r"C:\tri_disease_project\results\tables\node_table_with_clusters.csv")
id_to_symbol = node_table.set_index('name')['display name'].to_dict()

hubba['gene_symbol'] = hubba['node_name'].map(id_to_symbol)
print("\nUnmapped rows:", hubba['gene_symbol'].isna().sum())
print(hubba[['node_name', 'gene_symbol']].head(10))


Unmapped rows: 374
                      node_name gene_symbol
9606.ENSP00000383996        2.0         NaN
9606.ENSP00000233242   363517.0         NaN
9606.ENSP00000354947        2.0         NaN
9606.ENSP00000376902        1.0         NaN
9606.ENSP00000352798        1.0         NaN
9606.ENSP00000465356   362881.0         NaN
9606.ENSP00000318351        2.0         NaN
9606.ENSP00000358903       25.0         NaN
9606.ENSP00000389814     1347.0         NaN
9606.ENSP00000451300        2.0         NaN


In [9]:
import pandas as pd

path = r"C:\tri_disease_project\results\tables\cytohubba_all_scores.csv"  # adjust if your local path differs

# read raw and manually assign correct column names, accounting for the trailing empty field
col_names = ['node_name', 'MCC', 'DMNC', 'MNC', 'Degree', 'EPC', 'BottleNeck',
             'EcCentricity', 'Closeness', 'Radiality', 'Betweenness', 'Stress',
             'ClusteringCoefficient', 'extra']

hubba = pd.read_csv(path, header=0, names=col_names, index_col=False)
hubba = hubba.drop(columns=['extra'])

print("Shape:", hubba.shape)
print(hubba.head())
print("\nnode_name sample:", hubba['node_name'].head(5).tolist())

Shape: (374, 13)
              node_name       MCC     DMNC   MNC  Degree     EPC  BottleNeck  \
0  9606.ENSP00000383996       2.0  0.00000   1.0     2.0   4.666         2.0   
1  9606.ENSP00000233242  363517.0  0.60089  15.0    16.0  85.134        11.0   
2  9606.ENSP00000354947       2.0  0.00000   1.0     2.0   2.294         2.0   
3  9606.ENSP00000376902       1.0  0.00000   1.0     1.0  18.021         1.0   
4  9606.ENSP00000352798       1.0  0.00000   1.0     1.0   1.857         1.0   

   EcCentricity  Closeness  Radiality  Betweenness   Stress  \
0       0.11111   77.25595    7.82842    744.00000   3676.0   
1       0.11111  128.69802    9.64879   1904.09198  11382.0   
2       0.10000   67.58611    7.11260    145.86009    284.0   
3       0.12500  115.86786    9.51743      0.00000      0.0   
4       0.10000   63.48452    6.83110      0.00000      0.0   

   ClusteringCoefficient  
0                    0.0  
1                    0.5  
2                    0.0  
3              

In [10]:
node_table = pd.read_csv(r"C:\tri_disease_project\results\tables\node_table_with_clusters.csv")
id_to_symbol = node_table.set_index('name')['display name'].to_dict()

hubba['gene_symbol'] = hubba['node_name'].map(id_to_symbol)
print("Unmapped rows:", hubba['gene_symbol'].isna().sum())
print(hubba[['node_name', 'gene_symbol']].head(10))

# save the fully mapped table
hubba.to_csv(r"C:\tri_disease_project\results\tables\cytohubba_mapped.csv", index=False)

Unmapped rows: 0
              node_name gene_symbol
0  9606.ENSP00000383996       CERT1
1  9606.ENSP00000233242        APOB
2  9606.ENSP00000354947      CAPZA2
3  9606.ENSP00000376902        BANP
4  9606.ENSP00000352798     COL18A1
5  9606.ENSP00000465356       APOC1
6  9606.ENSP00000318351      BCKDHB
7  9606.ENSP00000358903     CYP17A1
8  9606.ENSP00000389814      ADIPOQ
9  9606.ENSP00000451300        NUMB


In [11]:
TOP_N = 20  # top 20 genes per method — standard choice in hub-gene papers

methods = ['Degree', 'MCC', 'MNC', 'Betweenness', 'Closeness']

top_sets = {}
for m in methods:
    top_genes = hubba.nlargest(TOP_N, m)['gene_symbol'].tolist()
    top_sets[m] = set(top_genes)
    print(f"\nTop {TOP_N} by {m}:")
    print(top_genes)

# count how many methods each gene appears as a "top hub" in
from collections import Counter
all_top_genes = [g for s in top_sets.values() for g in s]
hub_counts = Counter(all_top_genes)

hub_consensus = pd.DataFrame(hub_counts.items(), columns=['gene_symbol', 'n_methods_top_hub'])
hub_consensus = hub_consensus.sort_values('n_methods_top_hub', ascending=False).reset_index(drop=True)

print("\n\n=== Consensus hub genes (appearing in top", TOP_N, "for multiple methods) ===")
print(hub_consensus.to_string(index=False))

# defensible hub gene list: appears in top-N for at least 3 of the 5 methods
defensible_hubs = hub_consensus[hub_consensus['n_methods_top_hub'] >= 3]
print(f"\n\nDefensible hub genes (in >=3/5 methods' top {TOP_N}):", len(defensible_hubs))
print(defensible_hubs['gene_symbol'].tolist())

hub_consensus.to_csv(r"C:\tri_disease_project\results\tables\hub_gene_consensus.csv", index=False)
defensible_hubs.to_csv(r"C:\tri_disease_project\results\tables\hub_genes_final.csv", index=False)


Top 20 by Degree:
['TP53', 'TNF', 'IL6', 'ESR1', 'PPARG', 'APOE', 'PTEN', 'IGF1', 'BCL2', 'CCND1', 'CD4', 'PIK3R1', 'LPL', 'ADCY5', 'LEP', 'PPARGC1A', 'BDNF', 'SMAD3', 'IGF2', 'PLCG1']

Top 20 by MCC:
['APOE', 'LPL', 'APOB', 'APOA1', 'CETP', 'LIPC', 'APOC1', 'APOA5', 'APOH', 'LPA', 'IL6', 'PPARG', 'ADIPOQ', 'TP53', 'TNF', 'BCL2', 'IGF1', 'LEP', 'ESR1', 'SCARB1']

Top 20 by MNC:
['TP53', 'IL6', 'TNF', 'ESR1', 'APOE', 'IGF1', 'PPARG', 'BCL2', 'CCND1', 'PTEN', 'LPL', 'LEP', 'BDNF', 'CD4', 'APOB', 'ACE', 'IGF2', 'PIK3R1', 'PPARGC1A', 'ERBB4']

Top 20 by Betweenness:
['TP53', 'GNAS', 'PPARG', 'ESR1', 'TNF', 'ADCY5', 'DMD', 'PTEN', 'PIK3R1', 'SMAD3', 'APOE', 'IL6', 'CD4', 'PLCG1', 'GRM5', 'LPL', 'ADRB2', 'CCND1', 'GATA6', 'KCNJ11']

Top 20 by Closeness:
['TP53', 'PPARG', 'ESR1', 'IL6', 'TNF', 'IGF1', 'BCL2', 'PTEN', 'CCND1', 'CD4', 'APOE', 'PPARGC1A', 'NR3C1', 'LEP', 'ERBB4', 'PIK3R1', 'GNAS', 'BDNF', 'HSPA4', 'SMAD3']


=== Consensus hub genes (appearing in top 20 for multiple methods) ===

In [12]:
import pandas as pd

# paste the MCODE results as a dict: cluster number -> list of STRING protein IDs
mcode_clusters = {
    1: "9606.ENSP00000299022, 9606.ENSP00000233242, 9606.ENSP00000321334, 9606.ENSP00000252486, 9606.ENSP00000497642, 9606.ENSP00000465356, 9606.ENSP00000200676, 9606.ENSP00000205948, 9606.ENSP00000236850, 9606.ENSP00000445002",
    2: "9606.ENSP00000399168, 9606.ENSP00000339398, 9606.ENSP00000364080, 9606.ENSP00000353099, 9606.ENSP00000365402",
    3: "9606.ENSP00000308339, 9606.ENSP00000350364, 9606.ENSP00000299766, 9606.ENSP00000303222, 9606.ENSP00000384347",
    4: "9606.ENSP00000325822, 9606.ENSP00000358903, 9606.ENSP00000379683, 9606.ENSP00000451406, 9606.ENSP00000292427",
    5: "9606.ENSP00000356438, 9606.ENSP00000380252, 9606.ENSP00000225577, 9606.ENSP00000381185, 9606.ENSP00000376943, 9606.ENSP00000484824, 9606.ENSP00000287820, 9606.ENSP00000384371, 9606.ENSP00000415743, 9606.ENSP00000384018, 9606.ENSP00000228872, 9606.ENSP00000361021, 9606.ENSP00000332973, 9606.ENSP00000302961, 9606.ENSP00000262662, 9606.ENSP00000261693, 9606.ENSP00000310928, 9606.ENSP00000303706, 9606.ENSP00000269305",
    6: "9606.ENSP00000257068, 9606.ENSP00000393312, 9606.ENSP00000391826, 9606.ENSP00000370473, 9606.ENSP00000342235, 9606.ENSP00000345708, 9606.ENSP00000410242, 9606.ENSP00000486891, 9606.ENSP00000376637, 9606.ENSP00000282728, 9606.ENSP00000274695",
    7: "9606.ENSP00000490107, 9606.ENSP00000390600, 9606.ENSP00000328968, 9606.ENSP00000356265",
    8: "9606.ENSP00000351957, 9606.ENSP00000282249, 9606.ENSP00000296518, 9606.ENSP00000282096",
    9: "9606.ENSP00000306997, 9606.ENSP00000228841, 9606.ENSP00000442795, 9606.ENSP00000499406",
    10: "9606.ENSP00000426983, 9606.ENSP00000267101, 9606.ENSP00000353731, 9606.ENSP00000362353, 9606.ENSP00000261523, 9606.ENSP00000385780, 9606.ENSP00000356713, 9606.ENSP00000011653, 9606.ENSP00000244007, 9606.ENSP00000264867, 9606.ENSP00000312652, 9606.ENSP00000327943, 9606.ENSP00000443631, 9606.ENSP00000384517, 9606.ENSP00000428056, 9606.ENSP00000227507, 9606.ENSP00000371790, 9606.ENSP00000447378, 9606.ENSP00000385675",
    11: "9606.ENSP00000414303, 9606.ENSP00000398698, 9606.ENSP00000231509",
    12: "9606.ENSP00000349723, 9606.ENSP00000254480, 9606.ENSP00000386593",
    13: "9606.ENSP00000401946, 9606.ENSP00000399797, 9606.ENSP00000379144",
    14: "9606.ENSP00000343515, 9606.ENSP00000369695, 9606.ENSP00000378578",
    15: "9606.ENSP00000301921, 9606.ENSP00000313921, 9606.ENSP00000371923",
    16: "9606.ENSP00000354504, 9606.ENSP00000286744, 9606.ENSP00000476145",
    17: "9606.ENSP00000428780, 9606.ENSP00000276282, 9606.ENSP00000308318",
    18: "9606.ENSP00000242729, 9606.ENSP00000371512, 9606.ENSP00000354923",
    19: "9606.ENSP00000390436, 9606.ENSP00000296575, 9606.ENSP00000332353",
    20: "9606.ENSP00000219833, 9606.ENSP00000386069, 9606.ENSP00000280155",
    21: "9606.ENSP00000294016, 9606.ENSP00000419361, 9606.ENSP00000347046",
    22: "9606.ENSP00000398832, 9606.ENSP00000360141, 9606.ENSP00000305372",
}

mcode_scores = {1:10, 2:5, 3:5, 4:5, 5:4.556, 6:4.4, 7:4, 8:4, 9:3.333, 10:3.333,
                11:3, 12:3, 13:3, 14:3, 15:3, 16:3, 17:3, 18:3, 19:3, 20:3, 21:3, 22:3}

node_table = pd.read_csv(r"C:\tri_disease_project\results\tables\node_table_with_clusters.csv")
id_to_symbol = node_table.set_index('name')['display name'].to_dict()

records = []
for cluster_id, id_string in mcode_clusters.items():
    ids = [x.strip() for x in id_string.split(",")]
    symbols = [id_to_symbol.get(i, f"UNMAPPED:{i}") for i in ids]
    records.append({
        'cluster': cluster_id,
        'score': mcode_scores[cluster_id],
        'n_nodes': len(ids),
        'genes': ", ".join(symbols)
    })

mcode_df = pd.DataFrame(records)
pd.set_option('display.max_colwidth', None)
print(mcode_df.to_string(index=False))

mcode_df.to_csv(r"C:\tri_disease_project\results\tables\mcode_clusters_mapped.csv", index=False)

 cluster  score  n_nodes                                                                                                                                  genes
       1 10.000       10                                                                            LIPC, APOB, LPA, APOE, LPL, APOC1, CETP, APOH, APOA1, APOA5
       2  5.000        5                                                                                             HLA-B, HLA-DQA1, HLA-DQB1, HLA-DRB1, HLA-C
       3  5.000        5                                                                                                     SEC16B, NEGR1, MC4R, MTCH2, TMEM18
       4  5.000        5                                                                                            CYP11B2, CYP17A1, CYP19A1, HSD17B6, CYP11B1
       5  4.556       19 PTGS2, NFE2L2, RPS6KB1, BCL2, BCL2L11, MUC1, PPARG, BCL6, CD36, NCOR2, CDKN1B, PTEN, SMAD3, HSPA4, CDKN2C, SCARB1, PPARD, CDC25A, TP53
       6  4.400       11                

In [13]:
print("=== PHASE 4 SUMMARY ===")
print("Full network (from 690 candidate genes, 686 mapped): 686 nodes, 1012 edges, 289 connected components")
print("Largest Connected Component (LCC): 374 nodes, 983 edges, 1 connected component")
print("Defensible hub genes (>=3/5 CytoHubba methods, top 20 each):", len(defensible_hubs))
print(defensible_hubs['gene_symbol'].tolist())
print("\nMCODE modules found: 22")
print("Top thematic modules:")
print(" - Cluster 1 (lipoprotein metabolism):", mcode_df.loc[mcode_df.cluster==1, 'genes'].values[0])
print(" - Cluster 4 (steroidogenesis):", mcode_df.loc[mcode_df.cluster==4, 'genes'].values[0])
print(" - Cluster 6 (insulin/IGF/T2D signaling):", mcode_df.loc[mcode_df.cluster==6, 'genes'].values[0])
print(" - Cluster 10 (circadian/incretin axis):", mcode_df.loc[mcode_df.cluster==10, 'genes'].values[0])
print(" - Cluster 22 (RAAS/adrenergic):", mcode_df.loc[mcode_df.cluster==22, 'genes'].values[0])

=== PHASE 4 SUMMARY ===
Full network (from 690 candidate genes, 686 mapped): 686 nodes, 1012 edges, 289 connected components
Largest Connected Component (LCC): 374 nodes, 983 edges, 1 connected component
Defensible hub genes (>=3/5 CytoHubba methods, top 20 each): 17
['PPARG', 'TNF', 'ESR1', 'IL6', 'TP53', 'APOE', 'PTEN', 'BCL2', 'IGF1', 'LEP', 'LPL', 'CCND1', 'CD4', 'PIK3R1', 'SMAD3', 'PPARGC1A', 'BDNF']

MCODE modules found: 22
Top thematic modules:
 - Cluster 1 (lipoprotein metabolism): LIPC, APOB, LPA, APOE, LPL, APOC1, CETP, APOH, APOA1, APOA5
 - Cluster 4 (steroidogenesis): CYP11B2, CYP17A1, CYP19A1, HSD17B6, CYP11B1
 - Cluster 6 (insulin/IGF/T2D signaling): MTNR1B, FGFR1, IGF2, IGFBP3, ERBB4, KCNJ11, IGF2BP2, TCF7L2, IGF1, HHEX, CDKAL1
 - Cluster 10 (circadian/incretin axis): CLOCK, ERBB3, DPP4, GLP1R, RORA, DGKB, IFNGR1, CD4, PLCG1, PPARGC1A, LEP, SLC5A2, PLCB3, ARNTL, PIK3R1, CCND1, FGF9, MGAM, IL6
 - Cluster 22 (RAAS/adrenergic): AGTR1, GNAS, ADRB2
